In [1]:
import pandas as pd
import numpy as np


In [2]:
# Load Online Retail dataset
data = pd.read_csv("data.csv", encoding="latin1")

print(data.head())
print(data.info())


  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

      InvoiceDate  UnitPrice  CustomerID         Country  
0  12/1/2010 8:26       2.55     17850.0  United Kingdom  
1  12/1/2010 8:26       3.39     17850.0  United Kingdom  
2  12/1/2010 8:26       2.75     17850.0  United Kingdom  
3  12/1/2010 8:26       3.39     17850.0  United Kingdom  
4  12/1/2010 8:26       3.39     17850.0  United Kingdom  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo   

In [27]:
import pandas as pd

data = pd.read_csv("data.csv", encoding="latin1")

print(data.head())
print(data.columns)


  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

      InvoiceDate  UnitPrice  CustomerID         Country  
0  12/1/2010 8:26       2.55     17850.0  United Kingdom  
1  12/1/2010 8:26       3.39     17850.0  United Kingdom  
2  12/1/2010 8:26       2.75     17850.0  United Kingdom  
3  12/1/2010 8:26       3.39     17850.0  United Kingdom  
4  12/1/2010 8:26       3.39     17850.0  United Kingdom  
Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='object')


In [24]:
print(data.columns)


Index(['customer_id', 'product_id', 'timestamp', 'Quantity'], dtype='object')


In [3]:
# Keep only required columns
data = data[['CustomerID', 'StockCode', 'InvoiceDate', 'Quantity']]

# Rename columns for recommendation system
data.rename(columns={
    'CustomerID': 'customer_id',
    'StockCode': 'product_id',
    'InvoiceDate': 'timestamp'
}, inplace=True)


In [4]:
# Remove missing customer IDs
data.dropna(subset=['customer_id'], inplace=True)

# Remove cancelled / returned items
data = data[data['Quantity'] > 0]

# Convert customer_id to integer
data['customer_id'] = data['customer_id'].astype(int)

# Remove duplicates
data.drop_duplicates(inplace=True)


In [5]:
data['timestamp'] = pd.to_datetime(data['timestamp'])


In [6]:
user_item_interactions = (
    data
    .groupby(['customer_id', 'product_id'])['Quantity']
    .sum()
    .reset_index()
)

print(user_item_interactions.head())


   customer_id product_id  Quantity
0        12346      23166     74215
1        12347      16008        24
2        12347      17021        36
3        12347      20665         6
4        12347      20719        40


In [7]:
user_item_matrix = user_item_interactions.pivot(
    index='customer_id',
    columns='product_id',
    values='Quantity'
).fillna(0)

print(user_item_matrix.head())
print("Matrix Shape:", user_item_matrix.shape)


product_id   10002  10080  10120  10123C  10124A  10124G  10125  10133  10135  \
customer_id                                                                     
12346          0.0    0.0    0.0     0.0     0.0     0.0    0.0    0.0    0.0   
12347          0.0    0.0    0.0     0.0     0.0     0.0    0.0    0.0    0.0   
12348          0.0    0.0    0.0     0.0     0.0     0.0    0.0    0.0    0.0   
12349          0.0    0.0    0.0     0.0     0.0     0.0    0.0    0.0    0.0   
12350          0.0    0.0    0.0     0.0     0.0     0.0    0.0    0.0    0.0   

product_id   11001  ...  90214V  90214W  90214Y  90214Z  BANK CHARGES   C2  \
customer_id         ...                                                      
12346          0.0  ...     0.0     0.0     0.0     0.0           0.0  0.0   
12347          0.0  ...     0.0     0.0     0.0     0.0           0.0  0.0   
12348          0.0  ...     0.0     0.0     0.0     0.0           0.0  0.0   
12349          0.0  ...     0.0     0.0   

In [8]:
user_item_interactions.to_csv("clean_user_item_interactions.csv", index=False)
user_item_matrix.to_csv("user_item_matrix.csv")


Milestone 2

In [9]:
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD


In [10]:
user_item_matrix = pd.read_csv("user_item_matrix.csv", index_col=0)
print(user_item_matrix.shape)
user_item_matrix.head()


(4339, 3665)


,10002,10080,10120,10123C,10124A,10124G,10125,10133,10135,11001,...,90214V,90214W,90214Y,90214Z,BANK CHARGES,C2,DOT,M,PADS,POST
customer_id,,,,,,,,,,,,,,,,,,,,,
12346,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12347,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12348,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0
12349,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
12350,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [11]:
train_matrix = user_item_matrix.copy()
test_matrix = pd.DataFrame(0, index=user_item_matrix.index, columns=user_item_matrix.columns)

np.random.seed(42)  # reproducibility

for user in user_item_matrix.index:
    purchased_items = user_item_matrix.loc[user]
    purchased_items = purchased_items[purchased_items > 0].index.tolist()

    if len(purchased_items) < 2:
        continue  # skip users with only 1 purchase

    # Randomly select 1 item for test
    test_item = np.random.choice(purchased_items, size=1)[0]

    # Move it from train to test
    train_matrix.at[user, test_item] = 0
    test_matrix.at[user, test_item] = user_item_matrix.at[user, test_item]

print("Train matrix shape:", train_matrix.shape)
print("Test matrix shape:", test_matrix.shape)


Train matrix shape: (4339, 3665)
Test matrix shape: (4339, 3665)


In [12]:
n_components = 20

svd = TruncatedSVD(n_components=n_components, random_state=42)
svd.fit(train_matrix.values)

user_factors = svd.transform(train_matrix.values)
item_factors = svd.components_

# Reconstruct predicted interactions
predicted_matrix = np.dot(user_factors, item_factors)
predicted_df = pd.DataFrame(predicted_matrix,
                            index=train_matrix.index,
                            columns=train_matrix.columns)

predicted_df.head()


,10002,10080,10120,10123C,10124A,10124G,10125,10133,10135,11001,...,90214V,90214W,90214Y,90214Z,BANK CHARGES,C2,DOT,M,PADS,POST
customer_id,,,,,,,,,,,,,,,,,,,,,
12346,-0.044445,-0.004964,0.000276,-4.458039e-05,-0.000417,-0.000259,-0.114709,0.113450,-0.049741,0.257785,...,-0.000148,-2.939560e-05,-0.000113,-2.939560e-05,-0.000264,0.102703,-0.001863,-1.111976,-0.000115,-0.116506
12347,0.265432,0.006549,0.005835,2.159419e-05,0.000301,0.000377,0.339386,0.211413,0.322624,0.746828,...,0.000071,3.791240e-05,0.000305,3.791240e-05,0.000052,0.233419,0.003351,34.442984,0.000242,0.499605
12348,0.026158,0.008746,-0.000274,1.199029e-04,0.000348,0.000423,0.030581,0.251003,0.584983,-0.148393,...,0.000241,7.034708e-05,0.000174,7.034708e-05,0.000316,0.322520,0.001653,-9.875110,0.000398,-0.020714
12349,0.008339,0.000480,0.000773,2.462325e-06,0.000024,0.000015,0.015059,0.008688,0.004764,0.005746,...,0.000006,-7.229690e-07,0.000005,-7.229690e-07,0.000023,0.003915,0.000308,-0.408435,0.000006,0.058515
12350,0.009394,0.000262,0.000268,7.103835e-07,0.000014,0.000008,0.014429,0.006415,0.014518,-0.005530,...,0.000005,9.178520e-07,0.000007,9.178520e-07,0.000006,0.008125,0.000205,0.516164,-0.000001,0.003730


In [13]:
top_n = 5
recommendations = {}

for user_idx, user_id in enumerate(train_matrix.index):
    user_pred = predicted_matrix[user_idx]

    # Mask already purchased items in train
    purchased_mask = train_matrix.iloc[user_idx].values > 0
    user_pred[purchased_mask] = -np.inf

    # Top-N recommended product IDs
    top_items = train_matrix.columns[np.argsort(-user_pred)][:top_n]
    recommendations[user_id] = top_items.tolist()

# Sample recommendations
for u in list(recommendations.keys())[:5]:
    print(f"User {u}: Recommended Products: {recommendations[u]}")


User 12346: Recommended Products: ['17084R', '21169', '21166', '23230', '21668']
User 12347: Recommended Products: ['21212', 'M', '22630', '22629', '22326']
User 12348: Recommended Products: ['21212', '21975', '15036', '22659', '84077']
User 12349: Recommended Products: ['84077', '21977', '23084', '22629', '22630']
User 12350: Recommended Products: ['21891', '22440', '22693', '22086', '22492']


In [14]:
hits = 0
total_test_items = 0

for user_id in test_matrix.index:
    test_items = test_matrix.loc[user_id]
    test_items = test_items[test_items > 0].index.tolist()

    if len(test_items) == 0:
        continue

    total_test_items += len(test_items)

    # Check if test item is in recommended top-N
    for item in test_items:
        if item in recommendations[user_id]:
            hits += 1

print(f"Hit Rate @ {top_n}: {hits}/{total_test_items} = {hits/total_test_items:.4f}")


Hit Rate @ 5: 107/4247 = 0.0252


In [15]:
recs_df = pd.DataFrame([(k, v) for k, v in recommendations.items()],
                       columns=['customer_id', 'recommended_products'])
recs_df.to_csv("top_n_recommendations.csv", index=False)


Milestone 3

In [16]:
def precision_recall_f1_at_n(predictions, test_matrix, N=5):
    """
    predictions: dict {user_id: list of top-N recommended product_ids}
    test_matrix: DataFrame with hidden test items
    N: number of recommendations

    Returns: overall precision, recall, F1
    """
    precisions = []
    recalls = []

    for user_id in test_matrix.index:
        test_items = test_matrix.loc[user_id]
        test_items = test_items[test_items > 0].index.tolist()

        if len(test_items) == 0:
            continue

        recommended_items = predictions.get(user_id, [])

        hits = len(set(recommended_items) & set(test_items))

        precisions.append(hits / N)
        recalls.append(hits / len(test_items))

    precision = np.mean(precisions)
    recall = np.mean(recalls)

    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * (precision * recall) / (precision + recall)

    return precision, recall, f1


In [17]:
N = 5  # Top-N recommendations
precision, recall, f1 = precision_recall_f1_at_n(recommendations, test_matrix, N)

print(f"Top-{N} Recommendations Metrics:")
print(f"Precision@{N}: {precision:.4f}")
print(f"Recall@{N}: {recall:.4f}")
print(f"F1@{N}: {f1:.4f}")


Top-5 Recommendations Metrics:
Precision@5: 0.0050
Recall@5: 0.0252
F1@5: 0.0084


In [18]:
latent_factors = [10, 20, 30, 50]
best_f1 = 0
best_n_factors = 0
best_recommendations = {}

for n in latent_factors:
    svd = TruncatedSVD(n_components=n, random_state=42)
    svd.fit(train_matrix.values)

    user_factors = svd.transform(train_matrix.values)
    item_factors = svd.components_
    predicted_matrix = np.dot(user_factors, item_factors)

    # Generate Top-N recommendations
    temp_recommendations = {}
    for user_idx, user_id in enumerate(train_matrix.index):
        user_pred = predicted_matrix[user_idx]
        purchased_mask = train_matrix.iloc[user_idx].values > 0
        user_pred[purchased_mask] = -np.inf
        top_items = train_matrix.columns[np.argsort(-user_pred)][:N]
        temp_recommendations[user_id] = top_items.tolist()

    # Evaluate
    _, _, f1_score = precision_recall_f1_at_n(temp_recommendations, test_matrix, N)
    print(f"Latent Factors: {n}, F1@{N}: {f1_score:.4f}")

    if f1_score > best_f1:
        best_f1 = f1_score
        best_n_factors = n
        best_recommendations = temp_recommendations

print(f"Best Latent Factors: {best_n_factors}, Best F1@{N}: {best_f1:.4f}")


Latent Factors: 10, F1@5: 0.0059
Latent Factors: 20, F1@5: 0.0084
Latent Factors: 30, F1@5: 0.0090
Latent Factors: 50, F1@5: 0.0121
Best Latent Factors: 50, Best F1@5: 0.0121


In [19]:
for N in [5, 10, 15]:
    precision, recall, f1 = precision_recall_f1_at_n(best_recommendations, test_matrix, N)
    print(f"Top-{N} | Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")


Top-5 | Precision: 0.0073, Recall: 0.0363, F1: 0.0121
Top-10 | Precision: 0.0036, Recall: 0.0363, F1: 0.0066
Top-15 | Precision: 0.0024, Recall: 0.0363, F1: 0.0045


In [20]:
# Most active users (purchased > 5 items)
active_users = train_matrix[train_matrix.sum(axis=1) > 5].index

precision_active, recall_active, f1_active = precision_recall_f1_at_n(
    {u: best_recommendations[u] for u in active_users},
    test_matrix.loc[active_users],
    N=5
)

print(f"Most active users | Precision@5: {precision_active:.4f}, Recall@5: {recall_active:.4f}, F1@5: {f1_active:.4f}")


Most active users | Precision@5: 0.0072, Recall@5: 0.0362, F1@5: 0.0121


In [21]:
recs_df = pd.DataFrame([(k, v) for k, v in best_recommendations.items()],
                       columns=['customer_id', 'recommended_products'])
recs_df.to_csv("top_n_recommendations_refined.csv", index=False)


Milestone 4

In [28]:
product_details = (
    data[['StockCode', 'Description', 'UnitPrice']]
    .drop_duplicates()
    .reset_index(drop=True)
)

product_details.head()


,StockCode,Description,UnitPrice
0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2.55
1,71053,WHITE METAL LANTERN,3.39
2,84406B,CREAM CUPID HEARTS COAT HANGER,2.75
3,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,3.39
4,84029E,RED WOOLLY HOTTIE WHITE HEART.,3.39


In [29]:
data = data.dropna(subset=['CustomerID'])
data = data[data['Quantity'] > 0]
data = data[data['UnitPrice'] > 0]

data['CustomerID'] = data['CustomerID'].astype(int)


In [31]:
import pickle

pickle.dump(svd, open('svd_model.pkl', 'wb'))
pickle.dump(user_item_matrix, open('user_item_matrix.pkl', 'wb'))
pickle.dump(product_details, open('product_details.pkl', 'wb'))


In [32]:
import pandas as pd

# Load your dataset (make sure you are using the correct encoding)
data = pd.read_csv("data.csv", encoding="latin1")

# Remove missing CustomerID rows and ensure integers
data = data.dropna(subset=['CustomerID'])
data['CustomerID'] = data['CustomerID'].astype(int)

# Show 10 unique customer IDs
customer_ids = data['CustomerID'].unique()[:10]
print("Sample 10 Customer IDs:")
print(customer_ids)


Sample 10 Customer IDs:
[17850 13047 12583 13748 15100 15291 14688 17809 15311 14527]
